# Reformat & cleaning

1. Missing value problem
 - missing buyer_name -> di sesuikan sama lspe id cht 127 = jakarta
 - missing mainprocurementcategory = "Unknown Category"
 - trim juga sama kek training

2. Bad data
 - wajib award_value, dan award_value <= 0 gone
 - null

# 1. Define library & path

In [52]:
import pandas as pd
import numpy as np

file_path = f'../data/csv/master_data_ocds.csv'

data = pd.read_csv(file_path)

print(f"Loaded raw data {len(data)} rows, {len(data.columns)} columns")
print("\n--- Initial Missing Values ---")
print(data.isnull().sum().to_string())

Loaded raw data 21156 rows, 18 columns

--- Initial Missing Values ---
lspe_id                       0
nama_daerah                   0
ocid                          0
release_id                    0
date                          0
buyer_name                 3438
buyer_id                   2026
tender_id                     0
tender_title                  0
mainProcurementCategory       0
tender_minValue               0
tender_datePublished          0
tender_status                 0
award_id                      0
award_title                   0
award_date                    0
award_value                   0
award_supplier                0


# 2. STANDARISASI NAMA KOLOM

In [53]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
print("Columns:", data.columns.tolist())

Columns: ['lspe_id', 'nama_daerah', 'ocid', 'release_id', 'date', 'buyer_name', 'buyer_id', 'tender_id', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_datepublished', 'tender_status', 'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier']


# 3. PARSING TIPE DATA (date & number)

In [54]:
# Parse dates (membuang timezone agar aman untuk Machine Learning)
date_cols = ['date', 'tender_datepublished', 'award_date']
for col in date_cols:
    data[col] = pd.to_datetime(data[col], utc=True, errors='coerce').dt.tz_localize(None)

# Cast numerics (memastikan nilai uang berupa angka murni)
num_cols = ['tender_minvalue', 'award_value']
for col in num_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

print("Tipe data setelah dikonversi (Dtypes):")
print(data[date_cols + num_cols].dtypes.to_string(), "\n")

Tipe data setelah dikonversi (Dtypes):
date                    datetime64[us]
tender_datepublished    datetime64[us]
award_date              datetime64[us]
tender_minvalue                float64
award_value                    float64 



# 4. Handel Missing Data & Bad value 

In [55]:
before = len(data)

# duplicates
data.drop_duplicates(inplace=True)
afterDup = len(data)

print(f"Duplicate Rows removed: {before - afterDup}")

# incomplete data
beforeAw = len(data)
data.dropna(subset=['award_value', 'award_supplier'], inplace=True)
afterAw = len(data)
print(f"Awards Rows removed: {beforeAw - afterAw}")

# corr
beforeCor = len(data)
data = data[data['award_value'] > 0]
afterCor = len(data)
print(f"Corup Rows removed: {beforeCor - afterCor}")

after = len(data)
print(f"Rows removed: {before - after} ({before} -> {after})")


Duplicate Rows removed: 0
Awards Rows removed: 0
Corup Rows removed: 0
Rows removed: 0 (21156 -> 21156)


In [56]:
# buyer_name: try to recover from other rows sharing the same buyer_id
buyer_map = (
    data.dropna(subset=['buyer_name', 'buyer_id'])
        .drop_duplicates('buyer_id')
        .set_index('buyer_id')['buyer_name']
        .to_dict()
)
data['buyer_name'] = data['buyer_name'].fillna(data['buyer_id'].map(buyer_map))

# Remaining buyer_name nulls fill with 'Unknown Buyer'
data['buyer_name'] = data['buyer_name'].fillna('Pemerintah Daerah Provinsi Dki Jakarta')

# mainprocurementcategory fill with 'Unknown Category'
data['mainprocurementcategory'] = data['mainprocurementcategory'].fillna('Unknown Category')

# tender_minvalue fill missing with median (reasonable estimate)
median_min = data['tender_minvalue'].median()
data['tender_minvalue'] = data['tender_minvalue'].fillna(median_min)

print("Missing values after imputation:")
print(data.isnull().sum().to_string())

Missing values after imputation:
lspe_id                       0
nama_daerah                   0
ocid                          0
release_id                    0
date                          0
buyer_name                    0
buyer_id                   2026
tender_id                     0
tender_title                  0
mainprocurementcategory       0
tender_minvalue               0
tender_datepublished          0
tender_status                 0
award_id                      0
award_title                   0
award_date                    0
award_value                   0
award_supplier                0


In [57]:
text_cols = ['tender_title', 'award_title', 'buyer_name', 'award_supplier',
             'mainprocurementcategory', 'tender_status']
for col in text_cols:
    data[col] = data[col].astype(str).str.strip().str.title()

print("Sample buyer_name values:")
print(data['buyer_name'].value_counts().head(10))

Sample buyer_name values:
buyer_name
Pemerintah Daerah Provinsi Dki Jakarta    10574
Pemerintah Daerah Provinsi Jawa Tengah     5068
Pemerintah Daerah Provinsi Jawa Timur      5034
Badan Pengembangan Wilayah Suramadu         100
Badan Penanggulangan Lumpur Sidoarjo         54
Komite Olah Raga Nasional Indonesia          52
Sekretariat Kabinet                          35
Kementerian Kesehatan                        33
Pt Surabaya Industrial Estate Rungkut        33
Kementerian Perhubungan                      26
Name: count, dtype: int64


In [58]:
data['award_year']  = data['award_date'].dt.year
data['award_month'] = data['award_date'].dt.month

# Extract year and month from tender published date
data['publish_year']  = data['tender_datepublished'].dt.year
data['publish_month'] = data['tender_datepublished'].dt.month

# Calculate days_to_award: days between tender publish and award date
data['days_to_award'] = (data['award_date'] - data['tender_datepublished']).dt.days

# Clip negative values to 0
data['days_to_award'] = data['days_to_award'].clip(lower=0)

print("New temporal features:")
print(data[['award_year', 'award_month', 'publish_year', 'publish_month', 'days_to_award']].describe())

New temporal features:
         award_year   award_month  publish_year  publish_month  days_to_award
count  21156.000000  21156.000000  21156.000000   21156.000000   21156.000000
mean    2017.988703      6.322273   2017.956939       5.792730      28.138779
std        2.451055      2.837327      2.511743       3.031608     186.084554
min     2014.000000      1.000000   1970.000000       1.000000       0.000000
25%     2016.000000      4.000000   2016.000000       3.000000      15.000000
50%     2018.000000      6.000000   2018.000000       6.000000      22.000000
75%     2019.000000      9.000000   2019.000000       8.000000      35.000000
max     2024.000000     12.000000   2024.000000      12.000000   19156.000000


In [59]:
# budget_utilization_ratio: how much of the minimum tender budget was used
# Clip to avoid division by zero; cap ratio at 10x to remove extreme outliers
data['budget_utilization_ratio'] = (
    data['award_value'] / data['tender_minvalue'].replace(0, np.nan)
).clip(upper=10)
data['budget_utilization_ratio'] = data['budget_utilization_ratio'].fillna(1.0)

print("budget_utilization_ratio stats:")
print(data['budget_utilization_ratio'].describe())

budget_utilization_ratio stats:
count    21156.000000
mean         0.883579
std          0.097605
min          0.040056
25%          0.820036
50%          0.903046
75%          0.958601
max          1.997665
Name: budget_utilization_ratio, dtype: float64


In [60]:
# Drop only true identity/noise columns
# tender_datepublished is dropped because it's been fully extracted into publish_year, publish_month, days_to_award
cols_to_drop = ['ocid', 'release_id', 'buyer_id', 'award_id', 'tender_datepublished', 'tender_id', 'award_year', 'award_month', 'publish_year', 'publish_month']

data.drop(columns=[c for c in cols_to_drop if c in data.columns], inplace=True)

print("Remaining columns:")
print(data.columns.tolist())

Remaining columns:
['lspe_id', 'nama_daerah', 'date', 'buyer_name', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_status', 'award_title', 'award_date', 'award_value', 'award_supplier', 'days_to_award', 'budget_utilization_ratio']


In [61]:
print("=" * 40)
print("FINAL DATASET SUMMARY")
print("=" * 40)
print(f"Total rows   : {len(data)}")
print(f"Total columns: {len(data.columns)}")
print("\nMissing values (should all be 0):")
print(data.isnull().sum().to_string())
print("\nColumn dtypes:")
print(data.dtypes.to_string())
print("\nSample data:")
data.head(3)

FINAL DATASET SUMMARY
Total rows   : 21156
Total columns: 14

Missing values (should all be 0):
lspe_id                     0
nama_daerah                 0
date                        0
buyer_name                  0
tender_title                0
mainprocurementcategory     0
tender_minvalue             0
tender_status               0
award_title                 0
award_date                  0
award_value                 0
award_supplier              0
days_to_award               0
budget_utilization_ratio    0

Column dtypes:
lspe_id                              int64
nama_daerah                            str
date                        datetime64[us]
buyer_name                             str
tender_title                           str
mainprocurementcategory                str
tender_minvalue                    float64
tender_status                          str
award_title                            str
award_date                  datetime64[us]
award_value                        flo

,lspe_id,nama_daerah,date,buyer_name,tender_title,mainprocurementcategory,tender_minvalue,tender_status,award_title,award_date,award_value,award_supplier,days_to_award,budget_utilization_ratio
0,127,Jakarta,2014-12-24,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Makanan Binatang/Satwa Tmr (Pakan Ke...,Goods,1.499502e+09,Complete,Pengadaan Makanan Binatang/Satwa Tmr (Pakan Ke...,2014-12-24,1.157888e+09,Cv. Usaha Selaras,16,0.772182
1,127,Jakarta,2015-04-24,Pemerintah Daerah Provinsi Dki Jakarta,Pengadaan Makanan Binatang/ Satwa Tmr (Buah Da...,Goods,9.039069e+09,Complete,Pengadaan Makanan Binatang/ Satwa Tmr (Buah Da...,2015-04-24,6.867945e+09,Cv. Pangan Indo,115,0.759807
2,127,Jakarta,2015-04-23,Pemerintah Daerah Provinsi Dki Jakarta,Belanja Bahan Pakan Daging/Binatang Hidup,Goods,6.271924e+09,Complete,Belanja Bahan Pakan Daging/Binatang Hidup,2015-04-23,6.196783e+09,Cv. Rosadia Petaho,113,0.988019


## Train / Test Split (85/15)

In [ ]:
import os
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    data,
    test_size=0.15,
    train_size=0.85,
    random_state=42
)

print(f"Total rows    : {len(data)}")
print(f"Training rows : {len(train_data)} ({len(train_data)/len(data)*100:.1f}%)")
print(f"Testing rows  : {len(test_data)} ({len(test_data)/len(data)*100:.1f}%)")

# --- KODE TAMBAHAN UNTUK MENYIMPAN DATA ---
output_dir = '../data/cleaned'

# PERBAIKAN: Buat sub-folder 'train' dan 'test' secara eksplisit
os.makedirs(f"{output_dir}/train", exist_ok=True)
os.makedirs(f"{output_dir}/test", exist_ok=True)

train_path = f"{output_dir}/train/train_data.csv"
test_path = f"{output_dir}/test/test_data.csv"

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)

print("\n--- Penyimpanan Berhasil ---")
print(f"Data latih tersimpan di : {train_path}")
print(f"Data uji tersimpan di   : {test_path}")

Total rows    : 21156
Training rows : 17982 (85.0%)
Testing rows  : 3174 (15.0%)

--- Penyimpanan Berhasil ---
✅ Data latih tersimpan di : ../data/cleaned/train/train_data.csv
✅ Data uji tersimpan di   : ../data/cleaned/test/test_data.csv
